# Load and run a GGUF model in Java

This notebook creates a deterministic nano model locally, then exercises the same GGUF loader, memory mapping, forward pass, generation loop, and diagnostics used for downloaded models. It performs no model download.

In [1]:
import com.integrallis.models.api.SamplingOptions;
import com.integrallis.models.backend.purejava.PureJavaBackend;
import com.integrallis.models.runtime.GenerationLoop;
import com.integrallis.models.test.NanoGgufModel;
import java.nio.file.Files;
import java.nio.file.Path;

## Create an exact model artifact

A real application would point to an installed ModelJar artifact. The nano fixture keeps this example fast and deterministic while preserving the actual GGUF boundary.

In [2]:
Path fixtureDirectory = Files.createTempDirectory("models-notebook-");
Path modelPath = NanoGgufModel.write(fixtureDirectory.resolve("models-nano.gguf"));
System.out.println("fixture exists: " + Files.isRegularFile(modelPath));
System.out.println("fixture bytes: " + Files.size(modelPath));

fixture exists: true


fixture bytes: 25120


## Load, inspect, and generate

`PureJavaBackend` owns the mapped model resources, so it is closed with try-with-resources. The generation loop uses greedy sampling for a repeatable result.

In [3]:
try (var backend = PureJavaBackend.load(modelPath)) {
    System.out.println("model: " + backend.metadata().modelName());
    System.out.println("family: " + backend.metadata().modelFamily());
    System.out.println("vocabulary: " + backend.tokenizer().vocabSize());

    float[] logits = backend.forward(5, 0);
    System.out.println("logits: " + logits.length);
    System.out.println("backend: " + backend.diagnostics().backend());

    var options = SamplingOptions.builder()
        .temperature(0.0f)
        .topK(1)
        .maxTokens(4)
        .build();
    var generation = new GenerationLoop(backend).generate("t5", options);
    System.out.println("generated token text: "
        + (generation.isEmpty() ? "<end-of-generation>" : generation));
}

model: ModelsNano


family: llama


vocabulary: 32


logits: 32


backend: pure-java


generated token text: t17t5t22t15


In [4]:
Files.deleteIfExists(modelPath);
Files.deleteIfExists(fixtureDirectory);
System.out.println("fixture cleaned: true");

fixture cleaned: true
